<a href="https://colab.research.google.com/github/Fatima-05/Task1_News_Topic_Classifier/blob/main/Task1_News_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Problem statement and goal:

# build a ML model to classify news headlines into categories: World, Sports, Business, Sci/Tech
# learn how to preprocess text data and use a pretrained BERT transformer for sequence classification
# understand tokenization, attention masks, and embedding representations for NLP
# evaluate the model using metrics like accuracy, F1 score, and confusion matrix
# deploy the trained model using Streamlit to allow interactive classification of user input

In [ ]:
!pip install transformers datasets evaluate scikit-learn matplotlib seaborn -q
!pip install -U transformers datasets evaluate

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate

from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
dataset = load_dataset("ag_news")

dataset

In [ ]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print(train_df.head())

print("Class distribution:")
print(train_df.label.value_counts())

In [ ]:
# defining human-readable label names and creating mappings between label IDs and names
labels = ["World", "Sports", "Business", "Sci/Tech"]

id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

In [ ]:
# initalizing BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# defining a function to tokenize headlines
def tokenize_function(example):

    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
# tokenizing dataset, renaming cloumn to labels, and setting format for PyTorch
tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

tokenized_dataset.set_format("torch")

In [ ]:
# loadding a pretrained BERT model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

In [ ]:
# loadding accuracy, F1 metrics and defining compute_metrics function for evaluation
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [ ]:
# defining training hyperparameters
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True
)

In [ ]:
# initializing Hugging Face trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].shuffle(seed=42).select(range(8000)),
    eval_dataset=tokenized_dataset["test"].select(range(4000)),
    compute_metrics=compute_metrics
)

In [ ]:
# Finetunning the model on the training set and extracting final evaluation metrics
trainer.train()

results = {}
for log_entry in trainer.state.log_history:
    if 'eval_loss' in log_entry:
        results = {k: v for k, v in log_entry.items() if k.startswith('eval_')}
        break

print(results)

In [ ]:
# generatig predictions on the test dataset
predictions = trainer.predict(tokenized_dataset["test"].select(range(4000)))

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=labels))

In [ ]:
# visualizing the predicted vs actual classes using cm heatmap
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=labels,
            yticklabels=labels)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
# saving the finetuned model and tokenizer
model.save_pretrained("news_classifier_model")
tokenizer.save_pretrained("news_classifier_model")

In [ ]:
# Results and final insights:

# the AG News dataset was successfully loaded and preprocessed into tokenized inputs for BERT
# a pretrained BERT-base model was fine-tuned for 2 epochs on 8000 training samples and evaluated on 4000 test samples
# evaluation metrics include weighted F1-score, accuracy, and a confusion matrix, showing strong classification performance
# the model captures general trends and can classify headlines into the four main categories
# the Streamlit app provides an interactive interface for classifying new headlines, with predicted labels and confidence scores
# this workflow demonstrates a full NLP pipeline from data preprocessing to model deployment